In [1]:
import torch
import torch.nn as nn

In [ ]:
block_size = 8 # n-1 gram
embedding_size = 16
head_size = 512
batch_size = 512
steps = 20000

In [3]:
# input & output initialization
with open("inputs-shakespeare.txt", "r", encoding="utf-8") as f:
    data = f.read()

chars = sorted(set(data))
vocab_size = len(chars)
ctoi = {char: i for i, char in enumerate(chars)}
itoc = dict(enumerate(chars))

split = int(len(data) * 0.9)
train, val = data[:split], data[split:]

x_train = torch.tensor([ctoi[ch] for ch in train[:-1]], dtype=torch.long).unfold(0, block_size, 1)
y_train = torch.tensor([ctoi[ch] for ch in train[block_size:]], dtype=torch.long)

x_val = torch.tensor([ctoi[ch] for ch in val[:-1]], dtype=torch.long).unfold(0, block_size, 1)
y_val = torch.tensor([ctoi[ch] for ch in val[block_size:]], dtype=torch.long)

In [ ]:
# model declaration
class MultiLayerPerceptron(nn.Module):
    def __init__(self, block_size, vocab_size, embedding_size, head_size):
        super().__init__()
        self.head = nn.Sequential(
            nn.Embedding(vocab_size, embedding_size),
            nn.Flatten(),
            nn.Linear(in_features=embedding_size * block_size, out_features=head_size),
            nn.ReLU(),
            nn.Linear(in_features=head_size, out_features=vocab_size)
        )

    def forward(self, input):
        return self.head(input)

In [ ]:
# model initialization
model = MultiLayerPerceptron(block_size, vocab_size, embedding_size, head_size)
# optimizer initialization
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)

In [6]:
# training
for step in range(steps):
    optimizer.zero_grad()

    batch = torch.randint(0, len(x_train), (batch_size,))
    train_logits = model(x_train[batch])
    train_loss = nn.functional.cross_entropy(train_logits, y_train[batch])

    if step % 1000 == 0 or step == steps - 1:
        with torch.no_grad():
            val_logits = model(x_val)
            val_loss = nn.functional.cross_entropy(val_logits, y_val)
            print(f"Step {step}: train loss = {train_loss.item()}, val loss = {val_loss.item()}")

    train_loss.backward()
    optimizer.step()

Step 0: train loss = 4.204757213592529, val loss = 4.209372520446777
Step 1000: train loss = 1.6560970544815063, val loss = 1.9041637182235718
Step 2000: train loss = 1.6360855102539062, val loss = 1.8277090787887573
Step 3000: train loss = 1.5514330863952637, val loss = 1.8073982000350952
Step 4000: train loss = 1.511910080909729, val loss = 1.786550521850586
Step 5000: train loss = 1.4743499755859375, val loss = 1.762213110923767
Step 6000: train loss = 1.4881927967071533, val loss = 1.7637916803359985
Step 7000: train loss = 1.461784839630127, val loss = 1.7426410913467407
Step 8000: train loss = 1.42679762840271, val loss = 1.7424575090408325
Step 9000: train loss = 1.4480127096176147, val loss = 1.7410929203033447
Step 10000: train loss = 1.5381838083267212, val loss = 1.726989507675171
Step 11000: train loss = 1.5322694778442383, val loss = 1.7242611646652222
Step 12000: train loss = 1.3899916410446167, val loss = 1.7209460735321045
Step 13000: train loss = 1.447270393371582, val

In [7]:
# prediction
with torch.inference_mode():
    output = ['\n'] * block_size
    max_output_length = 1000
    while len(output) < max_output_length:
        i = torch.tensor([[ctoi[ch] for ch in output[-block_size:]]], dtype=torch.long)
        
        logits = model(i)
        P = torch.softmax(logits, dim=1)

        j = torch.multinomial(P, num_samples=1, replacement=True).item()

        output.append(itoc[j])

In [8]:
print(''.join(output[block_size:]))

YoRKINIU:
We soversetle might of the poor of he, this tribelby-magwent abong inter certhis proughbuga, a then of kings abother;
What will me afore, murdererough, there,
As moon you son joys, that Cliffor'd the fully some mother speak!

RIVERS:
Procause to fin a kinspecassispow
Cill lamuse thy brows on:
In nought?

ESNrO Warwick she prices--should not to speak the prayer take bust thou father's good,
To feed?

GLOUCESTER:
No doubt,
You have of the shall not but cale me both like me place; alady both cry une tongmen be leniece
More trume.

ROMEO:
A cannot be for recorded geness To death talk of man awak on him wold
By were forgeder, for whore;
Ay, wo, that the king do blooke very night.

OXFORD:
What, good now do coliout, come.

LADY ANNE:
APaI Goodness.

CRINIUS:
I thank with thereon him:
You, no?

Ay, a villain priouse,
For fe live Cherene womanish is blowned.

AUFIDIUS:
All the trave?

LUCIO:
I were not
The Volsces for not you makese you thine;
No ro some morceful will:
Wherei
